In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Install required libraries
!pip install scikit-learn mlflow fastapi uvicorn scipy pandas numpy --quiet

print("Libraries installed. Please restart the runtime (Runtime -> Restart runtime...) if you encounter issues with newly installed packages.")

Libraries installed. Please restart the runtime (Runtime -> Restart runtime...) if you encounter issues with newly installed packages.


In [ ]:
# Install required libraries (run once)
!pip install scikit-learn mlflow fastapi uvicorn scipy pandas numpy --quiet

# Import standard libraries
import os                                    # for file system operations
import numpy as np                           # numerical arrays
import pandas as pd                          # tabular data handling
import warnings                              # to silence noisy warnings
warnings.filterwarnings("ignore")            # cleaner output in the notebook
print("Libraries imported ✓")                # sanity check
     # Import sklearn components for the pipeline
from sklearn.datasets import load_breast_cancer             # toy classification dataset
from sklearn.model_selection import train_test_split        # split data into train/test
from sklearn.preprocessing import StandardScaler            # feature scaling transform
from sklearn.linear_model import LogisticRegression         # simple classifier
from sklearn.pipeline import Pipeline                       # bundles transforms + model
from sklearn.metrics import accuracy_score, f1_score        # evaluation metrics

# Load the dataset (binary classification: malignant vs benign)
data = load_breast_cancer(as_frame=True)                    # returns a Bunch with pandas frame
X = data.data                                               # feature columns
y = data.target                                             # binary labels

# Split into train and test (80/20) with a fixed seed for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
    )
print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")  # check shapes


# Define the pipeline: scaling step + classifier step
pipeline = Pipeline([
    ("scaler", StandardScaler()),                           # step 1: standardize features
    ("clf",    LogisticRegression(max_iter=1000))           # step 2: logistic regression
])

# Fit the entire pipeline on training data (both steps learn together)
pipeline.fit(X_train, y_train)                              # one call trains the whole flow

# Predict on test set using the SAME pipeline (no skew possible)
y_pred = pipeline.predict(X_test)                           # uses the fitted scaler + model
# Evaluate the predictions
acc = accuracy_score(y_test, y_pred)                        # overall accuracy
f1  = f1_score(y_test, y_pred)                              # harmonic mean of precision/recall
print(f"Accuracy: {acc:.4f}")                               # show accuracy
print(f"F1-score: {f1:.4f}")                                # show F1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21

In [ ]:
import mlflow                                              # MLflow tracking library
import mlflow.sklearn                                      # sklearn-specific logging helpers

# Tell MLflow where to store experiment data (local folder)
mlflow.set_tracking_uri("file:./mlruns")                   # local file-based tracking server

# Create / select a named experiment (a logical group of runs)
mlflow.set_experiment("breast_cancer_classifier")          # auto-creates if not present
print("MLflow tracking set up ✓")                          # confirmation


# Try a few different hyperparameter settings and log each as a run
C_values = [0.01, 0.1, 1.0, 10.0]                          # regularization strengths to test

# Loop through each candidate hyperparameter
for C in C_values:                                          # iterate over the list
    with mlflow.start_run(run_name=f"logreg_C={C}"):       # context = one tracked run
# Build a fresh pipeline with this C value
        pipe = Pipeline([
            ("scaler", StandardScaler()),                  # same scaling step as before
            ("clf",    LogisticRegression(C=C, max_iter=1000))  # different regularization
        ])

        pipe.fit(X_train, y_train)                         # train on the training split
        preds = pipe.predict(X_test)                       # predict on the test split

        # Compute metrics
        acc = accuracy_score(y_test, preds)                # accuracy on test set
        f1  = f1_score(y_test, preds)                      # F1 on test set

        # Log hyperparameters to MLflow
        mlflow.log_param("C", C)                           # store the C value
        mlflow.log_param("model", "LogisticRegression")    # store the model family

        # Log metrics to MLflow
        mlflow.log_metric("accuracy", acc)                 # store the accuracy
        mlflow.log_metric("f1_score", f1)                  # store the F1

        # Log the trained pipeline as a reusable artifact
        mlflow.sklearn.log_model(pipe, "model")            # saves model files under the run

        print(f"C={C:>5} | acc={acc:.4f} | f1={f1:.4f}")   # progress print


ModuleNotFoundError: No module named 'mlflow'

In [ ]:
display(X.head())

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [ ]:
print(f"Best run ID: {best_run.info.run_id}")               # show its ID
print(f"Best F1:     {best_run.data.metrics['f1_score']:.4f}")  # show its F1
print(f"Best C:      {best_run.data.params['C']}")          # show its hyperparameter


# Register the best run's model under a friendly name
model_uri = f"runs:/{best_run.info.run_id}/model"           # URI pointing at the artifact
registered_name = "BreastCancerClassifier"                  # name in the registry

# Register the model — creates version 1, or version N+1 on subsequent runs
result = mlflow.register_model(model_uri, registered_name)  # adds to the registry
print(f"Registered '{registered_name}' as version {result.version}")  # confirmation


NameError: name 'best_run' is not defined

After restarting the runtime and re-executing the MLflow logging cells, the `all_runs` DataFrame should be populated. The following cell will then display its columns.

In [ ]:
import pandas as pd
import mlflow
import os

# Ensure MLflow allows file store
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

# Set the tracking URI and experiment name for MLflow
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("breast_cancer_classifier")

# Retrieve all runs from the experiment to populate all_runs
all_runs = mlflow.search_runs(experiment_names=["breast_cancer_classifier"])

print("Columns of all_runs DataFrame after (attempted) population:")

if not all_runs.empty:
    display(all_runs.columns)
else:
    print("all_runs DataFrame is still empty. Please ensure MLflow runs were logged successfully after runtime restart.")

Columns of all_runs DataFrame after (attempted) population:
all_runs DataFrame is still empty. Please ensure MLflow runs were logged successfully after runtime restart.


After restarting the runtime and re-executing the MLflow logging cells, the `all_runs` DataFrame should be populated. The following cell will then display its columns.

In [ ]:
import pandas as pd
import mlflow
import os

# Ensure MLflow allows file store
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

# Set the tracking URI and experiment name for MLflow
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("breast_cancer_classifier")

# Retrieve all runs from the experiment to populate all_runs
all_runs = mlflow.search_runs(experiment_names=["breast_cancer_classifier"])

print("Columns of all_runs DataFrame after (attempted) population:")

if not all_runs.empty:
    display(all_runs.columns)
else:
    print("all_runs DataFrame is still empty. Please ensure MLflow runs were logged successfully after runtime restart.")

Columns of all_runs DataFrame after (attempted) population:
all_runs DataFrame is still empty. Please ensure MLflow runs were logged successfully after runtime restart.


In [ ]:
try:
    import mlflow
    print("mlflow imported successfully!")
except ModuleNotFoundError:
    print("ModuleNotFoundError: mlflow is not found. Please ensure you have restarted the runtime after installing packages and executed the installation cell (e5aa4382).")

mlflow imported successfully!


In [ ]:
import mlflow
import os

# Ensure MLflow allows file store
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

# List all experiments
experiments = mlflow.list_experiments()

if experiments:
    print("MLflow Experiments:")
    for exp in experiments:
        print(f"- Experiment ID: {exp.experiment_id}, Name: {exp.name}, Artifact Location: {exp.artifact_location}")
else:
    print("No MLflow experiments found.")

AttributeError: module 'mlflow' has no attribute 'list_experiments'

In [ ]:
import mlflow
import os

# Allow MLflow to use the filesystem tracking backend
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

# Ensure MLflow is configured to the same tracking URI and experiment
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("breast_cancer_classifier")

# Retrieve all runs from the experiment
all_runs = mlflow.search_runs(experiment_names=["breast_cancer_classifier"])

print("Logged Metrics for All Runs:")
for index, run in all_runs.iterrows():
    print(f"\nRun ID: {run.run_id}")
    print(f"  Run Name: {run['tags.mlflow.runName']}")
    print(f"  C parameter: {run['params.C']}")
    print(f"  Accuracy: {run['metrics.accuracy']:.4f}")
    print(f"  F1-score: {run['metrics.f1_score']:.4f}")

2026/06/22 05:40:01 INFO mlflow.tracking.fluent: Experiment with name 'breast_cancer_classifier' does not exist. Creating a new experiment.


Logged Metrics for All Runs:


In [ ]:
print("Columns of all_runs DataFrame:")
display(all_runs.columns)

Columns of all_runs DataFrame:


Index(['run_id', 'experiment_id', 'status', 'artifact_uri', 'start_time',
       'end_time'],
      dtype='object')

In [ ]:
import mlflow
import os

# Allow MLflow to use the filesystem tracking backend
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

# Ensure MLflow is configured to the same tracking URI and experiment
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("breast_cancer_classifier")

# Search for runs, ordering by f1_score in descending order
# We limit to 1 to get only the top run
best_run = mlflow.search_runs(
    order_by=["metrics.f1_score DESC"],
    max_results=1
).iloc[0] # Get the first (and only) row of the result DataFrame

print("Found best run:")
print(f"  Run ID: {best_run.run_id}")
print(f"  F1-score: {best_run['metrics.f1_score']:.4f}")
print(f"  C parameter: {best_run['params.C']}")
print(f"  Artifact URI: {best_run.artifact_uri}")

# You can also get the Run object itself for more detailed information
best_mlflow_run = mlflow.get_run(best_run.run_id)

IndexError: single positional indexer is out-of-bounds

In [ ]:
fastapi_code = """
import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel
import mlflow.pyfunc # Used to load MLflow-logged models
import os

app = FastAPI()

class Features(BaseModel):
    values: list[float]

# --- Model Loading within the FastAPI app ---
# This ensures the model is loaded when the FastAPI app starts,
# making the application self-contained.

# Set MLflow tracking URI, assuming a local file store for development/demonstration.
# In production, this might point to a remote MLflow Tracking Server.
os.environ["MLFLOW_TRACKING_URI"] = "file:./mlruns"
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true" # Ensure local file store is allowed

try:
    # Search for the run with the highest F1 score in the experiment.
    # This assumes that MLflow runs have been successfully logged to "./mlruns".
    best_run_df = mlflow.search_runs(
        experiment_names=["breast_cancer_classifier"],
        order_by=["metrics.f1_score DESC"],
        max_results=1
    )

    if best_run_df.empty:
        raise ValueError("No MLflow runs found for 'breast_cancer_classifier' experiment. Please ensure runs are logged.")

    best_run_id = best_run_df.iloc[0].run_id
    model_uri = f"runs:/{best_run_id}/model"

    # Load the best model
    loaded_model = mlflow.pyfunc.load_model(model_uri)
    print(f"Successfully loaded model from MLflow URI: {model_uri}")

except Exception as e:
    print(f"Error loading model for FastAPI: {e}")
    print("FastAPI app will start, but prediction endpoint might not function correctly without a loaded model.")
    # As a fallback, or to allow the app to start even without a successful model load,
    # a dummy model is initialized. In a real deployment, you might exit here.
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    loaded_model = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression())])
    print("Initialized a dummy model. Predictions will not be accurate until a proper model is loaded.")


@app.get("/")
def read_root():
    return {"status": "ok"}

@app.post("/predict")
def predict(payload: Features):
    arr = np.array(payload.values).reshape(1, -1)
    # Use the loaded model for predictions
    pred = int(loaded_model.predict(arr)[0])
    prob = float(loaded_model.predict_proba(arr)[0, 1])
    return {"prediction": pred, "probability": prob}
"""

# Write the file
with open("app.py", "w") as f:
    f.write(fastapi_code)
print("Wrote app.py ✓")

Wrote app.py ✓


In [ ]:
!pip list

Package                                  Version
---------------------------------------- ------------------
absl-py                                  1.4.0
accelerate                               1.13.0
access                                   1.1.10.post3
affine                                   2.4.0
aiofiles                                 24.1.0
aiohappyeyeballs                         2.6.2
aiohttp                                  3.14.1
aiosignal                                1.4.0
aiosqlite                                0.22.1
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.12.0
alembic                                  1.18.4
altair                                   5.5.0
annotated-doc                            0.0.4
annotated-types                          0.7.0
antlr4-python3-runtime                   4.9.3
anyio                          

In [ ]:

from scipy.stats import ks_2samp                          # two-sample KS test

# Simulate "live" production data by copying X_test and shifting feature 0
X_live = X_test.copy()                                      # start from test data
X_live.iloc[:, 0] = X_live.iloc[:, 0] + 5.0                 # add an offset → drift!

# Pick a feature to monitor
feature_to_check = X_train.columns[0]                       # first feature

# Reference distribution (what we trained on)
ref  = X_train[feature_to_check]                            # training values
# Live distribution (what we're seeing now)
live = X_live[feature_to_check]                             # production-like values

# Run a Kolmogorov-Smirnov test: are the two samples from the same distribution?
stat, p_value = ks_2samp(ref, live)                         # statistic + p-value

print(f"Feature:    {feature_to_check}")                    # show feature name
print(f"KS stat:    {stat:.4f}")                            # KS test statistic
print(f"p-value:    {p_value:.6f}")                         # significance

# Alert if drift detected (p-value very small = significantly different)
if p_value < 0.05:                                          # 5% threshold
    print("⚠️  DRIFT DETECTED — consider retraining the model")
else:
    print("✅ No significant drift")                        # all good


# Generalize: check every feature and report any that drifted
drift_report = []                                           # collect results here

# Loop over all feature columns
for col in X_train.columns:                                 # one feature at a time
    s, p = ks_2samp(X_train[col], X_live[col])              # KS test per feature
    drift_report.append({"feature": col, "ks": s, "p": p, "drifted": p < 0.05})

# Turn into a DataFrame for easy reading
report_df = pd.DataFrame(drift_report)                      # tabular results
drifted   = report_df[report_df["drifted"]]                 # only the flagged ones
print(f"Total features checked: {len(report_df)}")          # how many we tested
print(f"Drifted features:       {len(drifted)}")            # how many fired
drifted.head()                                              # show the worst offenders


Feature:    mean radius
KS stat:    0.6441
p-value:    0.000000
⚠️  DRIFT DETECTED — consider retraining the model
Total features checked: 30
Drifted features:       2


,feature,ks,p,drifted
0,mean radius,0.644110,1.745624e-36,True
4,mean smoothness,0.141488,4.651818e-02,True
